# 🕹️ <span style=color:dodgerblue>LLM for video game knowledge assistance<span>

This notebook is used for explanation purpose. Once the `docker compose up` command
is executed the entire application is ready to be used.

## 📔 <span style=color:yellow>What is this notebook about?</span>
This notebook is intended to explain all the pipeline and provide an overview 
of the processes involved in the application (ingestion, monitoring, evaluation, etc).  
It's a complement to the [README.md](README.md) that shows the internal mechanisms
of the scripts.

In [ ]:
from llm import RAGClient
from opensearchpy import OpenSearch

## Opensearch client creation


In [ ]:
opensearch_client = OpenSearch(
    hosts=[{"host": "localhost", "port": 9200}],
    http_auth=("admin", "Opensearch16admin#"),
    use_ssl=False,
    verify_certs=False, 
    ssl_show_warn=False,
)

rag_client = RAGClient(opensearch_client)


## <span style=color:lightsalmon>⛁ Ingestion</span>

This part showcases the ingestion process and how It works. To avoid the long 
waiting time this part is entirely optional and is not necessary to proceed to 
the blocks of this notebook It's just used for explanation purpose.

### 🎮 <span style=color:darkorchid>IGDB ingestion</span>
> In order to proceed here, you must have and IGDB developer key

In [ ]:
wikipedia_ground_truth = evaluator.generate_ground_truth(
        "wikipedia", 3, 300, "data/wikipedia_ground_truth.csv"
    )

igdb_ground_truth = evaluator.generate_ground_truth(
        "igdb", 3, 300, "data/igdb_ground_truth.csv"
    )


### <span style=color:deepskyblue>🖹 Wikipedia ingestion</span>
We pull the wikipedia information from huggingface index. The data was updated 
on 


## 🧪 <span style=color:forestgreen>Evaluation</span>

Here we will use our `Evaluator` to execute each one of the steps:
1. Ground truth generation
2. Search evaluation & optimization.
    1. Perform the base evaluation itself.
    2. Optimize the boost values.
3. Tools use and final RAG's answer evaluation.

### 🎯 Ground truth generation

In order to evaluate the search quality and model's performance we must have 
querys and a target variable to use as evaluation method. In this case the target 
variable is the document id and the query will be a llm generated question based on
the target document. 

For instance, if the target document is about Mario Kart, the llm
will generate `n` questions related to the document topic.

> <span style=color:yellow>⚠ **Warning**</span>  
> The code blocks in this section create a very small dataset so as to
> not waste your tokens. You may create a very large dataset if you want

### 🔎 Search evaluation & optimization

We must evaluate our search functions. Do they return the relevant documents?
For this we will use two metrics

In [ ]:
from evaluation import Evaluator

: 

In [ ]:
evaluator = Evaluator(rag_client)

igdb_hr_score, igdb_mrr_score, igdb_x = evaluator.evaluate_search(index="igdb")
wikipedia_hr_score, wikipedia_mrr_score, wikipedia_x = evaluator.evaluate_search(
    index="wikipedia"
)

display(
    "--- IGDB search evaluation ---"
    f"Hit rate score: {igdb_hr_score}"
    f"Mean reciprocal rank score: {igdb_mrr_score}"
    "--- wikipedia search evaluation ---"
    f"Hit rate score: {wikipedia_hr_score}"
    f"Mean reciprocal rank score: {wikipedia_mrr_score}"
)

### 🔧 Tools and final RAG answer evaluation.

In [ ]:
judge = RAGClient(opensearch_client, model="gemma-4-31b-it")
evaluator.evaluate_agent(judge)

## 👁️ Monitoring